# Matched Virchow2 H&E feature extraction


In [ ]:
from __future__ import annotations
from pathlib import Path
import os
import sys

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "config" / "project.yaml").is_file()
             and (path / "src" / "pathology").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter within this repository.")
sys.path.insert(0, str(ROOT / "src"))
from pathology.config import load_paths
PATHS = load_paths(ROOT)
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")
os.environ.setdefault("HF_HOME", str(ROOT / ".cache" / "huggingface"))
os.environ.setdefault("HF_MODULES_CACHE", str(ROOT / ".cache" / "huggingface" / "modules"))
os.environ.setdefault("TORCH_HOME", str(ROOT / ".cache" / "torch"))


## Configuration


In [ ]:
from pathlib import Path
import json
import os
import time
import numpy as np
import pandas as pd
import tifffile
import zarr
from PIL import Image
PROJECT = ROOT
RUN = PATHS.run
CASE_ROOT = RUN / 'cases'
REGISTRATION = PATHS.registration / 'registration_pairs.csv'
QUICK_TEST = False
MAX_TEST_CASES = 1
MAX_TEST_PATCHES_PER_CASE = 8
BATCH_SIZE = 1
TARGET_MPP = 0.5
MODEL_ID = 'paige-ai/Virchow2'
MODEL_DIR = PATHS.virchow2_model
LOW_MEMORY = True
OUTPUT = RUN / 'feature_characterisation' / ('virchow2_he_smoke' if QUICK_TEST else 'virchow2_he')
CASE_OUTPUT = OUTPUT / 'cases'
CASE_OUTPUT.mkdir(parents=True, exist_ok=True)
assert CASE_ROOT.is_dir() and REGISTRATION.is_file()
registration = pd.read_csv(REGISTRATION)
registration = registration.drop_duplicates('case_id').set_index('case_id')
case_paths = sorted(CASE_ROOT.glob('*/features.parquet'))
assert case_paths
if QUICK_TEST:
    case_paths = case_paths[:MAX_TEST_CASES]


## Frozen Virchow2


In [ ]:
import torch
import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
from timm.layers import SwiGLUPacked
if not torch.cuda.is_available():
    raise RuntimeError('Virchow2 requires a CUDA GPU for this workflow.')
if not (MODEL_DIR / 'config.json').is_file() or not (MODEL_DIR / 'model.safetensors').is_file():
    raise FileNotFoundError(f'Incomplete local Virchow2 repository: {MODEL_DIR}')
try:
    model = timm.create_model('local-dir:' + MODEL_DIR.as_posix(), pretrained=True, mlp_layer=SwiGLUPacked, act_layer=torch.nn.SiLU).eval()
    if LOW_MEMORY:
        model = model.half()
    model = model.to('cuda')
except Exception as exc:
    raise RuntimeError('Local Virchow2 could not be loaded. Check model files and GPU memory. The existing KRONOS2 outputs remain unchanged.') from exc
transform = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))


## Matched H&E reader


In [ ]:
def full_mpp(series):
    page = series.pages[0]
    x_resolution = page.tags.get('XResolution')
    unit = page.tags.get('ResolutionUnit')
    if x_resolution is None or unit is None:
        raise ValueError('H&E TIFF lacks physical resolution tags')
    numerator, denominator = x_resolution.value
    pixels_per_unit = numerator / denominator
    unit_value = int(unit.value)
    if unit_value == 3:
        return 10000.0 / pixels_per_unit
    if unit_value == 2:
        return 25400.0 / pixels_per_unit
    raise ValueError(f'Unsupported TIFF resolution unit: {unit_value}')

class HEReader:

    def __init__(self, path):
        self.path = Path(path)
        self.tif = tifffile.TiffFile(self.path)
        series = self.tif.series[0]
        if series.axes != 'YXS' or series.shape[-1] < 3:
            raise ValueError(f'Expected RGB H&E image in {self.path}')
        self.full_shape = series.shape[:2]
        self.full_mpp = full_mpp(series)
        choices = []
        for index, level in enumerate(series.levels):
            scale_x = self.full_shape[1] / level.shape[1]
            scale_y = self.full_shape[0] / level.shape[0]
            if not np.isclose(scale_x, scale_y, rtol=0.01):
                continue
            level_mpp = self.full_mpp * scale_x
            choices.append((abs(np.log(level_mpp / TARGET_MPP)), index, scale_x, level_mpp))
        if not choices:
            raise ValueError(f'No valid H&E pyramid level in {self.path}')
        _, self.level, self.scale, self.level_mpp = min(choices)
        self.store = tifffile.imread(self.path, series=0, level=self.level, aszarr=True)
        self.array = zarr.open(self.store, mode='r')

    def read_patch(self, row):
        target_size = int(round(float(row.patch_size) * float(row.segmentation_mpp) / self.level_mpp))
        x0 = int(round(float(row.he_full_x) / self.scale))
        y0 = int(round(float(row.he_full_y) / self.scale))
        x1, y1 = (x0 + target_size, y0 + target_size)
        if x0 < 0 or y0 < 0 or x1 > self.array.shape[1] or (y1 > self.array.shape[0]):
            raise ValueError(f'Patch outside H&E bounds: {row.patch_id}')
        rgb = np.asarray(self.array[y0:y1, x0:x1, :3])
        if rgb.ndim != 3 or rgb.shape[-1] != 3:
            raise ValueError(f'Invalid H&E RGB patch: {row.patch_id}')
        return Image.fromarray(rgb.astype(np.uint8), mode='RGB')

    def close(self):
        self.store.close()
        self.tif.close()
META = ['case_id', 'wsi_name', 'roi_id', 'patch_id', 'patch_x', 'patch_y', 'patch_size', 'patch_stride', 'he_full_x', 'he_full_y', 'segmentation_mpp']


## Resumable extraction


In [ ]:
feature_columns = [f'he_feature_{i:04d}' for i in range(2560)]
summary = []
for case_path in case_paths:
    case_id = case_path.parent.name
    if case_id not in registration.index:
        raise KeyError(f'Missing registration record for {case_id}')
    he_path = Path(registration.loc[case_id, 'fixed'])
    if not he_path.is_file():
        raise FileNotFoundError(he_path)
    patches = pd.read_parquet(case_path, columns=META).sort_values(['roi_id', 'patch_y', 'patch_x']).reset_index(drop=True)
    if QUICK_TEST:
        patches = patches.head(MAX_TEST_PATCHES_PER_CASE)
    assert patches.patch_id.is_unique
    target = CASE_OUTPUT / f'{case_id}.parquet'
    if target.is_file():
        existing = pd.read_parquet(target, columns=['patch_id'])
        if len(existing) == len(patches) and set(existing.patch_id) == set(patches.patch_id):
            summary.append({'case_id': case_id, 'patches': len(patches), 'status': 'reused'})
            continue
    reader = HEReader(he_path)
    vectors = np.empty((len(patches), 2560), dtype=np.float32)
    started = time.time()
    try:
        for start in range(0, len(patches), BATCH_SIZE):
            block = patches.iloc[start:start + BATCH_SIZE]
            inputs = torch.stack([transform(reader.read_patch(row)) for row in block.itertuples(index=False)]).to('cuda')
            with torch.inference_mode(), torch.autocast(device_type='cuda', dtype=torch.float16):
                tokens = model(inputs)
                embedding = torch.cat([tokens[:, 0], tokens[:, 5:].mean(dim=1)], dim=1)
            if embedding.shape != (len(block), 2560):
                raise ValueError(f'Unexpected Virchow2 output shape: {tuple(embedding.shape)}')
            vectors[start:start + len(block)] = embedding.float().cpu().numpy()
            del inputs, tokens, embedding
    finally:
        reader.close()
    if not np.isfinite(vectors).all():
        raise ValueError(f'Non-finite H&E embedding in {case_id}')
    result = pd.concat([patches, pd.DataFrame(vectors, columns=feature_columns)], axis=1)
    result['he_model'] = MODEL_ID
    result['he_level'] = reader.level
    result['he_mpp'] = reader.level_mpp
    temporary = target.with_suffix('.parquet.partial')
    result.to_parquet(temporary, index=False)
    temporary.replace(target)
    summary.append({'case_id': case_id, 'patches': len(patches), 'status': 'extracted', 'seconds': round(time.time() - started, 1)})
summary_frame = pd.DataFrame(summary)
summary_frame.to_csv(OUTPUT / 'extraction_summary.csv', index=False)
expected = sum((len(pd.read_parquet(path, columns=['patch_id'])) for path in case_paths))
if not QUICK_TEST and summary_frame.patches.sum() != expected:
    raise AssertionError('H&E feature count does not match completed KRONOS2 patch count')
manifest = {'model': MODEL_ID, 'model_dir': str(MODEL_DIR), 'mode': 'smoke' if QUICK_TEST else 'full', 'cases': int(len(summary_frame)), 'patches': int(summary_frame.patches.sum()), 'feature_dim': 2560, 'target_mpp': TARGET_MPP}
(OUTPUT / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
summary_frame
